# 02 - Data Inventory

Author: Saige Mukherjee

Contact: mukherjeesaige@gmail.com //
https://www.linkedin.com/in/saige-mukherjee-0aba68281/

The analysis runs inside the Jupyter notebook. Run the notebook with credentialed access to generate the results.

If you want to run the notebook and generate the report:
- obtained credentialed access to MIMIC-IV via PhysioNet and BigQuery,
- create a GCP project and star the MIMIC-IV dataset ,
- enter the GCP project ID below and execute the program.

In [ ]:
PROJECT_ID = "" # enter your GCP project ID into the string

**Project:** Healthcare Operations Analysis using MIMIC-IV v3.1  
**Notebook purpose:** Build a clean inventory of the tables, columns, row counts, key identifiers, and patient-flow-relevant fields available in MIMIC-IV.

This notebook is intentionally descriptive. It does not attempt to answer an operational question yet. Its job is to document what data exists, how it is structured, which tables are relevant to hospital flow, and what limitations need to be respected before deeper analysis.

## 1. Why this notebook exists

Before measuring patient flow, bed burden, transfer timing, long-stay tails, or care-unit bottlenecks, we need to know exactly what the dataset contains.

MIMIC-IV is organized into modules. For this project, the main modules are:

- `hosp`: hospital-wide EHR data, including admissions, transfers, services, orders, labs, medication records, billing codes, and patient demographics.
- `icu`: ICU-specific data sourced from MetaVision, including ICU stays, charted events, inputs, outputs, and procedures.

The patient-flow backbone of this project is mostly in the `hosp` module, especially:

- `patients`
- `admissions`
- `transfers`
- `services`

The ICU module is still important because `icustays` is derived from `transfers`, which helps validate how ICU movement is represented.

**Important privacy note:** this project should publish code, methods, and aggregate outputs only. Do not publish row-level MIMIC-IV data or derived datasets that could expose sensitive information.

## 2. Setup

This notebook assumes access to MIMIC-IV v3.1 through Google BigQuery via the `physionet-data` project.

Expected BigQuery datasets:

- `physionet-data.mimiciv_3_1_hosp`
- `physionet-data.mimiciv_3_1_icu`

If your access is configured differently, update the constants below.

In [ ]:
# Install the BigQuery client library
!pip install google-cloud-bigquery

In [ ]:
# Core packages
import pandas as pd
import numpy as np
from google.cloud import bigquery

# Display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 120)

# BigQuery client
client = bigquery.Client(project=PROJECT_ID)

# Dataset constants
HOSP = "physionet-data.mimiciv_3_1_hosp"
ICU = "physionet-data.mimiciv_3_1_icu"
MIN_CELL_N = 11  # conservative project-level suppression threshold

print("Hosp dataset:", HOSP)
print("ICU dataset:", ICU)

## 3. Confirm available tables

First, list the tables available in the `hosp` and `icu` modules. This gives us a reproducible data inventory rather than relying on memory or documentation alone.

In [ ]:
def list_tables(dataset_id: str) -> pd.DataFrame:
    """Return a dataframe of tables in a BigQuery dataset."""
    dataset_ref = bigquery.DatasetReference.from_string(dataset_id)
    tables = list(client.list_tables(dataset_ref))
    return pd.DataFrame({
        "dataset": [dataset_id] * len(tables),
        "table_name": [t.table_id for t in tables],
        "full_table_id": [f"{dataset_id}.{t.table_id}" for t in tables]
    }).sort_values("table_name").reset_index(drop=True)

hosp_tables = list_tables(HOSP)
icu_tables = list_tables(ICU)

all_tables = pd.concat([hosp_tables, icu_tables], ignore_index=True)

display(hosp_tables)
display(icu_tables)

## 4. Table row counts

Row counts tell us which tables are small dimension tables, which are event-level tables, and which tables need careful query design to avoid unnecessary BigQuery cost.

For this project, very large event tables are usually not needed unless we are asking a specific clinical-process question. Patient-flow work can begin with smaller administrative tables.

In [ ]:
def row_count_query(full_table_id: str) -> str:
    return f"SELECT '{full_table_id}' AS full_table_id, COUNT(*) AS row_count FROM `{full_table_id}`"

queries = [row_count_query(t) for t in all_tables["full_table_id"]]
combined_query = " UNION ALL ".join(queries)

row_counts = client.query(combined_query).to_dataframe()
row_counts["table_name"] = row_counts["full_table_id"].str.split(".").str[-1]
# Extract the module from the full_table_id
row_counts["dataset_module"] = row_counts["full_table_id"].str.extract(r"mimiciv_3_1_(hosp|icu)")
row_counts = row_counts[["dataset_module", "table_name", "full_table_id", "row_count"]]
row_counts = row_counts.sort_values(["row_count"], ascending=[False]).reset_index(drop=True)

display(row_counts)

## 5. Column inventory

Next, collect column names and data types from BigQuery `INFORMATION_SCHEMA`. This gives us a searchable schema dictionary.

In [ ]:
def get_columns(dataset_id: str) -> pd.DataFrame:
    project, dataset = dataset_id.split(".")
    query = f"""
    SELECT
        table_schema AS dataset_name,
        table_name,
        ordinal_position,
        column_name,
        data_type,
        is_nullable
    FROM `{project}.{dataset}.INFORMATION_SCHEMA.COLUMNS`
    ORDER BY table_name, ordinal_position
    """
    return client.query(query).to_dataframe()

hosp_columns = get_columns(HOSP)
icu_columns = get_columns(ICU)

columns = pd.concat([hosp_columns, icu_columns], ignore_index=True)
columns["dataset_module"] = columns["dataset_name"].str.extract(r"mimiciv_3_1_(hosp|icu)")
columns = columns[["dataset_module", "table_name", "ordinal_position", "column_name", "data_type", "is_nullable"]]

display(columns.head(50))
display(columns[columns['dataset_module'] == 'icu'].head(50)) # Display head of ICU columns explicitly
print(f"Total columns inventoried: {len(columns):,}")

## 6. Identify key linkage columns

MIMIC-IV uses repeated identifiers to link data across tables.

The most important identifiers for this project are:

- `subject_id`: unique patient identifier.
- `hadm_id`: unique hospital admission identifier. Some rows can have `hadm_id` missing when data are outside an inpatient hospitalization.
- `transfer_id`: unique row-level transfer identifier in the `transfers` table.
- `stay_id`: ICU stay identifier used in the ICU module.

For patient flow, `subject_id`, `hadm_id`, `intime`, `outtime`, `careunit`, and `eventtype` are especially important.

In [ ]:
key_columns = [
    "subject_id", "hadm_id", "stay_id", "transfer_id",
    "intime", "outtime", "admittime", "dischtime", "deathtime",
    "careunit", "eventtype", "curr_service", "prev_service"
]

key_column_inventory = (
    columns[columns["column_name"].isin(key_columns)]
    .sort_values(["column_name", "dataset_module", "table_name"])
    .reset_index(drop=True)
)

display(key_column_inventory)

## 7. Patient-flow-relevant tables

This project does not need every table equally. The table below is a working map of relevance.

In [ ]:
flow_table_notes = pd.DataFrame([
    {
        "module": "hosp",
        "table": "patients",
        "project_role": "Patient demographics and deidentified time anchors.",
        "important_fields": "subject_id, gender, anchor_age, anchor_year, anchor_year_group, dod",
        "notes": "Use for age bands and approximate real-year grouping. Do not compare deidentified calendar years across patients directly."
    },
    {
        "module": "hosp",
        "table": "admissions",
        "project_role": "Hospitalization-level record.",
        "important_fields": "subject_id, hadm_id, admittime, dischtime, admission_type, admission_location, discharge_location, insurance, language, marital_status, race",
        "notes": "Useful for admission/discharge context, LOS, ED admission source, and discharge disposition."
    },
    {
        "module": "hosp",
        "table": "transfers",
        "project_role": "Core patient movement table.",
        "important_fields": "subject_id, hadm_id, transfer_id, eventtype, careunit, intime, outtime",
        "notes": "Main table for care-unit burden, transfer duration, long-stay tails, and patient-flow sequences."
    },
    {
        "module": "hosp",
        "table": "services",
        "project_role": "Clinical service assignment during hospitalization.",
        "important_fields": "subject_id, hadm_id, transfertime, prev_service, curr_service",
        "notes": "Useful for comparing physical/unit movement against service-line responsibility."
    },
    {
        "module": "icu",
        "table": "icustays",
        "project_role": "ICU stay summary derived from transfers.",
        "important_fields": "subject_id, hadm_id, stay_id, first_careunit, last_careunit, intime, outtime, los",
        "notes": "Useful for validating ICU movement and separating ICU-specific stays from broader hospital flow."
    },
    {
        "module": "hosp",
        "table": "poe",
        "project_role": "Provider orders.",
        "important_fields": "subject_id, hadm_id, poe_id, ordertime, order_type",
        "notes": "Potential future table for process-timing questions, but not needed for first-pass flow inventory."
    },
    {
        "module": "hosp",
        "table": "procedures_icd",
        "project_role": "Billed procedures.",
        "important_fields": "subject_id, hadm_id, seq_num, chartdate, icd_code, icd_version",
        "notes": "Useful for case-mix/procedure grouping. Timing is date-level and billing-derived, so be careful."
    },
    {
        "module": "hosp",
        "table": "diagnoses_icd",
        "project_role": "Billed diagnoses.",
        "important_fields": "subject_id, hadm_id, seq_num, icd_code, icd_version",
        "notes": "Useful for case-mix grouping; not a real-time clinical process table."
    },
])

display(flow_table_notes)

## 8. Validate linkage integrity for core tables

These checks help confirm that core identifiers behave as expected before deeper analysis.

The goal is not to prove the dataset is perfect. The goal is to make sure our assumptions are explicit.

In [ ]:
linkage_checks_query = f"""
WITH
admissions AS (
    SELECT COUNT(*) AS admissions_rows, COUNT(DISTINCT hadm_id) AS unique_hadm_id
    FROM `{HOSP}.admissions`
),
patients AS (
    SELECT COUNT(*) AS patient_rows, COUNT(DISTINCT subject_id) AS unique_subject_id
    FROM `{HOSP}.patients`
),
transfers AS (
    SELECT
        COUNT(*) AS transfer_rows,
        COUNT(DISTINCT transfer_id) AS unique_transfer_id,
        COUNT(DISTINCT subject_id) AS transfer_subjects,
        COUNT(DISTINCT hadm_id) AS transfer_hadm_ids,
        COUNTIF(hadm_id IS NULL) AS transfer_rows_missing_hadm_id,
        COUNTIF(outtime IS NULL) AS transfer_rows_missing_outtime
    FROM `{HOSP}.transfers`
),
icustays AS (
    SELECT
        COUNT(*) AS icustay_rows,
        COUNT(DISTINCT stay_id) AS unique_stay_id,
        COUNT(DISTINCT subject_id) AS icustay_subjects,
        COUNT(DISTINCT hadm_id) AS icustay_hadm_ids
    FROM `{ICU}.icustays`
)
SELECT * FROM admissions CROSS JOIN patients CROSS JOIN transfers CROSS JOIN icustays
"""

linkage_checks = client.query(linkage_checks_query).to_dataframe()
display(linkage_checks)

## 9. Inventory values in `transfers`

The `transfers` table is the most important table for this project. We need to understand:

- which care units exist,
- which `eventtype` values exist,
- how often transfer rows have missing times,
- how transfer durations behave at a basic level.

In [ ]:
transfer_eventtypes = client.query(f"""
SELECT
    eventtype,
    COUNT(*) AS n_rows,
    COUNT(DISTINCT subject_id) AS n_subjects,
    COUNT(DISTINCT hadm_id) AS n_hadm_ids
FROM `{HOSP}.transfers`
GROUP BY eventtype
HAVING COUNT(DISTINCT subject_id) >= {MIN_CELL_N}
   AND COUNT(DISTINCT hadm_id) >= {MIN_CELL_N}
ORDER BY n_rows DESC
""").to_dataframe()

display(transfer_eventtypes)

In [ ]:
careunit_inventory = client.query(f"""
SELECT
    careunit,
    COUNT(*) AS n_rows,
    COUNT(DISTINCT subject_id) AS n_subjects,
    COUNT(DISTINCT hadm_id) AS n_hadm_ids,
    COUNTIF(hadm_id IS NULL) AS rows_missing_hadm_id,
    COUNTIF(intime IS NULL) AS rows_missing_intime,
    COUNTIF(outtime IS NULL) AS rows_missing_outtime
FROM `{HOSP}.transfers`
GROUP BY careunit
HAVING COUNT(DISTINCT subject_id) >= {MIN_CELL_N}
   AND COUNT(DISTINCT hadm_id) >= {MIN_CELL_N}
ORDER BY n_rows DESC
""").to_dataframe()

display(careunit_inventory)

## 10. Basic transfer-duration inventory

This is not the main analysis yet. It is a data-quality and scale check.

A transfer duration is calculated as:

`outtime - intime`

Rows with missing `outtime` or negative/zero durations should be inspected before deciding how to handle them.

In [ ]:
transfer_duration_inventory = client.query(f"""
WITH transfer_durations AS (
    SELECT
        transfer_id,
        subject_id,
        hadm_id,
        eventtype,
        careunit,
        intime,
        outtime,
        TIMESTAMP_DIFF(outtime, intime, MINUTE) / 60.0 AS duration_hours
    FROM `{HOSP}.transfers`
    WHERE intime IS NOT NULL
      AND outtime IS NOT NULL
)
SELECT
    COUNT(*) AS n_rows_with_duration,
    COUNTIF(duration_hours < 0) AS n_negative_duration,
    COUNTIF(duration_hours = 0) AS n_zero_duration,
    COUNTIF(duration_hours > 0) AS n_positive_duration,
    APPROX_QUANTILES(duration_hours, 100)[OFFSET(50)] AS p50_hours,
    APPROX_QUANTILES(duration_hours, 100)[OFFSET(75)] AS p75_hours,
    APPROX_QUANTILES(duration_hours, 100)[OFFSET(90)] AS p90_hours,
    APPROX_QUANTILES(duration_hours, 100)[OFFSET(95)] AS p95_hours,
    APPROX_QUANTILES(duration_hours, 100)[OFFSET(99)] AS p99_hours,
    MAX(duration_hours) AS max_hours
FROM transfer_durations
""").to_dataframe()

display(transfer_duration_inventory)

## 11. Admission-level inventory

The `admissions` table provides hospital-level context for transfer rows. For operations analysis, the most immediately useful fields are admission type, admission location, discharge location, insurance, language, race, and hospital length of stay.

Be careful with demographic variables. Use them only when they serve a clear analytic purpose, and present aggregate results responsibly.

In [ ]:
admission_inventory = client.query(f"""
SELECT
    COUNT(*) AS n_admissions,
    COUNT(DISTINCT subject_id) AS n_subjects,
    COUNT(DISTINCT hadm_id) AS n_hadm_ids,
    COUNTIF(admittime IS NULL) AS missing_admittime,
    COUNTIF(dischtime IS NULL) AS missing_dischtime,
    APPROX_QUANTILES(TIMESTAMP_DIFF(dischtime, admittime, HOUR) / 24.0, 100)[OFFSET(50)] AS median_los_days,
    APPROX_QUANTILES(TIMESTAMP_DIFF(dischtime, admittime, HOUR) / 24.0, 100)[OFFSET(75)] AS p75_los_days,
    APPROX_QUANTILES(TIMESTAMP_DIFF(dischtime, admittime, HOUR) / 24.0, 100)[OFFSET(90)] AS p90_los_days,
    APPROX_QUANTILES(TIMESTAMP_DIFF(dischtime, admittime, HOUR) / 24.0, 100)[OFFSET(95)] AS p95_los_days
FROM `{HOSP}.admissions`
WHERE admittime IS NOT NULL
  AND dischtime IS NOT NULL
""").to_dataframe()

display(admission_inventory)

In [ ]:
for col in ["admission_type", "admission_location", "discharge_location", "insurance", "language", "race"]:
    print(f"\n--- {col} ---")
    df = client.query(f"""
    SELECT
        {col},
        COUNT(*) AS n_admissions,
        ROUND(100 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct_admissions
    FROM `{HOSP}.admissions`
    GROUP BY {col}
    HAVING COUNT(*) >= {MIN_CELL_N}
    ORDER BY n_admissions DESC
    LIMIT 25
    """).to_dataframe()
    display(df)

## 12. Service-line inventory

The `services` table can help distinguish physical location from clinical responsibility.

For example, a patient may be physically in one care unit while managed by a particular service. This can become important if long stays are driven by service-level patterns rather than unit-level patterns alone.

In [ ]:
services_inventory = client.query(f"""
SELECT
    curr_service,
    COUNT(*) AS n_service_rows,
    COUNT(DISTINCT subject_id) AS n_subjects,
    COUNT(DISTINCT hadm_id) AS n_hadm_ids
FROM `{HOSP}.services`
GROUP BY curr_service
HAVING COUNT(DISTINCT subject_id) >= {MIN_CELL_N}
   AND COUNT(DISTINCT hadm_id) >= {MIN_CELL_N}
ORDER BY n_service_rows DESC
""").to_dataframe()

display(services_inventory)

In [ ]:
service_transition_inventory = client.query(f"""
SELECT
    prev_service,
    curr_service,
    COUNT(*) AS n_transitions,
    COUNT(DISTINCT subject_id) AS n_subjects,
    COUNT(DISTINCT hadm_id) AS n_hadm_ids
FROM `{HOSP}.services`
WHERE prev_service IS NOT NULL
GROUP BY prev_service, curr_service
HAVING COUNT(DISTINCT subject_id) >= {MIN_CELL_N}
   AND COUNT(DISTINCT hadm_id) >= {MIN_CELL_N}
ORDER BY n_transitions DESC
LIMIT 50
""").to_dataframe()

display(service_transition_inventory)

## 13. ICU inventory

The `icustays` table summarizes ICU stays. It is derived from `transfers`, and consecutive transfers within ICU locations are merged into one `stay_id` for analytical convenience.

For this project, `icustays` is useful for validating ICU-specific movement, separating ICU from non-ICU flow, and comparing ICU burden against broader hospital burden.

In [ ]:
icu_inventory = client.query(f"""
SELECT
    first_careunit,
    last_careunit,
    COUNT(*) AS n_icu_stays,
    COUNT(DISTINCT subject_id) AS n_subjects,
    COUNT(DISTINCT hadm_id) AS n_hadm_ids,
    APPROX_QUANTILES(los, 100)[OFFSET(50)] AS median_los_days,
    APPROX_QUANTILES(los, 100)[OFFSET(75)] AS p75_los_days,
    APPROX_QUANTILES(los, 100)[OFFSET(90)] AS p90_los_days,
    APPROX_QUANTILES(los, 100)[OFFSET(95)] AS p95_los_days
FROM `{ICU}.icustays`
GROUP BY first_careunit, last_careunit
HAVING COUNT(DISTINCT subject_id) >= {MIN_CELL_N}
   AND COUNT(DISTINCT hadm_id) >= {MIN_CELL_N}
ORDER BY n_icu_stays DESC
""").to_dataframe()

display(icu_inventory)

## 14. Time deidentification and timeline rules

MIMIC-IV dates are shifted into the future on a patient-specific basis. Time intervals within a patient are preserved, but deidentified calendar dates are not comparable across different patients.

Project implications:

- It is valid to calculate durations within a patient encounter, such as transfer duration or hospital length of stay.
- It is valid to compare approximate real time periods using `anchor_year_group`.
- It is not valid to interpret two different patients with the same deidentified year as occurring in the same real calendar year.

For temporal trend analysis, use `anchor_year`, `anchor_year_group`, and admission dates carefully.

In [ ]:
anchor_inventory = client.query(f"""
SELECT
    anchor_year_group,
    COUNT(*) AS n_patients,
    COUNTIF(anchor_age = 91) AS n_patients_age_90_plus_grouped
FROM `{HOSP}.patients`
GROUP BY anchor_year_group
HAVING COUNT(*) >= {MIN_CELL_N}
ORDER BY anchor_year_group
""").to_dataframe()

display(anchor_inventory)

## 15. Create a focused project data dictionary

This is the lightweight data dictionary for the current project scope. It can be expanded as the analysis matures.

In [ ]:
project_data_dictionary = pd.DataFrame([
    ["patients", "subject_id", "Patient identifier", "Primary patient-level linkage key."],
    ["patients", "anchor_age", "Age at anchor year", "Use for age bands; age > 89 grouped as 91."],
    ["patients", "anchor_year", "Deidentified anchor year", "Used with anchor_year_group for approximate real-year alignment."],
    ["patients", "anchor_year_group", "Approximate true year group", "Useful for broad temporal cohorts."],
    ["admissions", "hadm_id", "Hospital admission identifier", "Primary hospitalization-level linkage key."],
    ["admissions", "admittime", "Admission time", "Use for hospital LOS and sequencing within patient."],
    ["admissions", "dischtime", "Discharge time", "Use for hospital LOS and discharge timing."],
    ["admissions", "admission_location", "Admission source", "Useful for ED-origin and transfer-in cohorts."],
    ["admissions", "discharge_location", "Discharge destination", "Useful for placement/disposition analyses."],
    ["transfers", "transfer_id", "Transfer-row identifier", "Unique row key for patient movement events."],
    ["transfers", "eventtype", "Transfer event type", "Used to distinguish admission, transfer, discharge-like movement rows."],
    ["transfers", "careunit", "Care unit/location label", "Core grouping field for unit burden and flow."],
    ["transfers", "intime", "Transfer start time", "Start of stay in a care unit."],
    ["transfers", "outtime", "Transfer end time", "End of stay in a care unit."],
    ["services", "curr_service", "Current clinical service", "Useful for service-line responsibility."],
    ["services", "transfertime", "Service transfer time", "Timing of service responsibility changes."],
    ["icustays", "stay_id", "ICU stay identifier", "ICU-level stay key derived from transfer records."],
    ["icustays", "los", "ICU length of stay in days", "Summary ICU LOS measure."],
], columns=["table", "column", "meaning", "project_use"])

display(project_data_dictionary)

## 16. Initial interpretation

This inventory establishes that the project can proceed using a staged approach:

1. Use `transfers` as the primary movement table.
2. Use `admissions` to contextualize each hospitalization.
3. Use `patients` only for age/time-anchor context when needed.
4. Use `services` to compare unit-level movement against clinical service responsibility.
5. Use `icustays` to validate and isolate ICU movement.
6. Defer large clinical event tables unless a specific process-timing question requires them.

The immediate next notebook should move from inventory to **cohort construction and data cleaning**. That notebook should define the transfer-duration cohort, handle missing or unusual times, create broad care-unit categories, and produce a clean base table for downstream analysis.

## References

- Johnson, A., Bulgarelli, L., Pollard, T., Gow, B., Moody, B., Horng, S., Celi, L. A., & Mark, R. (2024). *MIMIC-IV* (version 3.1). PhysioNet. https://doi.org/10.13026/kpb9-mt58
- Johnson, A. E. W., Bulgarelli, L., Shen, L. et al. *MIMIC-IV, a freely accessible electronic health record dataset*. Scientific Data 10, 1 (2023). https://doi.org/10.1038/s41597-022-01899-x
- Goldberger, A. et al. *PhysioBank, PhysioToolkit, and PhysioNet: Components of a new research resource for complex physiologic signals*. Circulation 101(23), e215–e220 (2000).